# dataset papeline

In [22]:
import os
import tarfile
from exceptiongroup import catch
import scipy
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from torch.utils.data import random_split


In [23]:
save_root = "flower_data"
tgz_path = os.path.join(save_root, "102flowers.tgz")
print("📦开始解压...")
with tarfile.open(tgz_path, 'r:gz') as tar:
    tar.extractall(path=save_root)
print("✅解压完成！")

📦开始解压...
✅解压完成！


In [24]:
train_transform = transforms.Compose([
    # 随机增强（每次取样本时随机生效，用来增加数据多样性）
    transforms.RandomHorizontalFlip(p=0.5),      # 水平翻转，50%概率翻转图片
    transforms.RandomRotation(degrees=10),        # 随机旋转，±10度以内
    transforms.ColorJitter(brightness=0.2),       # 亮度随机波动 ±20%

    # 基础预处理（训练、验证集共用这套标准化操作）
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ===================== 验证集transform：**无任何随机增强** =====================
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.225],
                         std=[0.229, 0.224, 0.225])
])



In [25]:
class OxfordFlowerDataset(Dataset):
    def __init__ (self ,root_dir,transform):
        super().__init__()
        self.root_dir = root_dir
        self.img_dir = os.path.join(root_dir,'jpg')
        self.transform = transform
        #load matlab lable
        label_mat = scipy.io.loadmat(os.path.join(root_dir,'imagelabels.mat'))
        self.labels = label_mat['labels'][0]-1
        self.error_log = []

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        """Load image with error handling"""
        try:
            # 正常读取
            img_name = f'image_{idx+1:05d}.jpg'
            img_path = os.path.join(self.img_dir, img_name)
            image = Image.open(img_path)

            # 检查文件是否损坏（verify 会关闭文件，所以要重新 open）
            image.verify()
            image = Image.open(img_path)

            # 跳过太小的图
            if image.size[0] < 32 or image.size[1] < 32:
                raise ValueError(f"Image too small: {image.size}")

            # 灰度图 / 其他模式 → RGB
            if image.mode != 'RGB':
                image = image.convert('RGB')

            # 取标签
            label = self.labels[idx]

            # 预处理
            if self.transform:
                image = self.transform(image)

            return image, label

        except Exception as e:
            # 记录错误，不崩溃
            self.error_log.append({
                'index': idx,
                'error': str(e),
                'path': img_path if 'img_path' in locals() else 'unknown'
            })
            print(f"Warning: Skipping corrupted image {idx}: {e}")

            # 递归尝试下一张（取模防止越界）
            next_idx = (idx + 1) % len(self)
            return self.__getitem__(next_idx)


In [19]:
def get_error_summary(self):
    """Review what went wrong after training"""
    if not self.error_log:
        print("No errors encountered - dataset is clean!")
    else:
        print(f"\nEncountered {len(self.error_log)} problematic images:")
        for error in self.error_log[:5]:  # Show first 5
            print(f"  Index {error['index']}: {error['error']}")
        if len(self.error_log) > 5:
            print(f"  ... and {len(self.error_log) - 5} more")


In [20]:


dataset = OxfordFlowerDataset('./flower_data')
#split into train/validation/test:70/15/15
train_size = int(0.7*len(dataset))
val_size = int(0.15*len(dataset))
test_size = len(dataset)-train_size-val_size

train_dataset, val_dataset ,test_dataset = random_split(
    dataset ,[train_size,val_size,test_size]
)


In [21]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=32,shuffle=False)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False)